In [5]:
# ============================================================
# MEJOR MODELO FINAL - V-JEPA2 T=8 FUSED MEAN + 7 MLPs + POWER MEAN q=0.5
# Entrada esperada: z_fused [1024]
# Salida: clase predicha
# ============================================================

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = Path("checkpoints/powermean7")

Q_POWER_MEAN = 0.5
ENCODER_DIM = 1024

CLASSES = [
    "Assemble system",
    "Consult sheets",
    "No action",
    "Picking in front",
    "Picking left",
    "Put down component",
    "Put down measuring rod",
    "Put down screwdriver",
    "Take component",
    "Take measuring rod",
    "Take screwdriver",
    "Turn sheets"
]

NUM_CLASSES = len(CLASSES)

class SimpleMLPClassifier(nn.Module):
    def __init__(self, encoder_dim=1024, num_classes=12, hidden_dim=512, dropout=0.35):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim // 2),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


MODEL_CONFIGS = [
    {
        "path": OUTPUT_DIR / "best_OFFICIAL_vjepa2_t8_fused_mean_mlp_plain.pt",
        "hidden_dim": 512,
        "dropout": 0.35
    },
    {
        "path": OUTPUT_DIR / "Exp5A_plain_MLP_h512_drop0.25_lr0.0001_ls0.05.pt",
        "hidden_dim": 512,
        "dropout": 0.25
    },
    {
        "path": OUTPUT_DIR / "Exp6A_mlp_seed11_h512_drop025_lr1e4_ls005.pt",
        "hidden_dim": 512,
        "dropout": 0.25
    },
    {
        "path": OUTPUT_DIR / "Exp6A_mlp_seed22_h512_drop030_lr1e4_ls005.pt",
        "hidden_dim": 512,
        "dropout": 0.30
    },
    {
        "path": OUTPUT_DIR / "Exp6A_mlp_seed33_h512_drop025_lr3e4_ls003.pt",
        "hidden_dim": 512,
        "dropout": 0.25
    },
    {
        "path": OUTPUT_DIR / "Exp6A_mlp_seed44_h256_drop025_lr1e4_ls005.pt",
        "hidden_dim": 256,
        "dropout": 0.25
    },
    {
        "path": OUTPUT_DIR / "Exp6A_mlp_seed55_h512_drop020_lr1e4_ls003.pt",
        "hidden_dim": 512,
        "dropout": 0.20
    }
]


def load_final_models():
    models = []

    for cfg in MODEL_CONFIGS:
        assert cfg["path"].exists(), f"No existe el checkpoint: {cfg['path']}"

        ckpt = torch.load(
            cfg["path"],
            map_location=DEVICE,
            weights_only=False
        )

        model = SimpleMLPClassifier(
            encoder_dim=ENCODER_DIM,
            num_classes=NUM_CLASSES,
            hidden_dim=cfg["hidden_dim"],
            dropout=cfg["dropout"]
        ).to(DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        models.append(model)

    print(f"Modelos cargados: {len(models)}")
    return models


FINAL_MODELS = load_final_models()


@torch.no_grad()
def predict_action_final(z_fused, top_k=3):
    """
    Predice una acción usando el mejor modelo final.

    Entrada:
        z_fused: embedding V-JEPA2 fused mean
                 shape [1024] o [B, 1024]

    Salida:
        clase final + probabilidad + top_k
    """

    if isinstance(z_fused, np.ndarray):
        z_fused = torch.tensor(z_fused, dtype=torch.float32)

    if not torch.is_tensor(z_fused):
        raise TypeError("z_fused debe ser torch.Tensor o np.ndarray")

    z_fused = z_fused.float()

    if z_fused.ndim == 1:
        z_fused = z_fused.unsqueeze(0)

    assert z_fused.shape[1] == ENCODER_DIM, f"Se esperaba [B, 1024], llegó {z_fused.shape}"

    z_fused = F.normalize(z_fused, p=2, dim=1)
    z_fused = z_fused.to(DEVICE)

    probs_list = []

    for model in FINAL_MODELS:
        logits = model(z_fused)
        probs = torch.softmax(logits, dim=1)
        probs_list.append(probs)

    probs_stack = torch.stack(probs_list, dim=0)  # [7, B, 12]

    # Power mean q=0.5
    probs_power = torch.mean(probs_stack ** Q_POWER_MEAN, dim=0) ** (1.0 / Q_POWER_MEAN)
    probs_final = probs_power / probs_power.sum(dim=1, keepdim=True)

    confs, preds = torch.max(probs_final, dim=1)
    top_probs, top_idxs = torch.topk(probs_final, k=top_k, dim=1)

    results = []

    for i in range(z_fused.shape[0]):
        pred_idx = int(preds[i].item())

        results.append({
            "predicted_class_id": pred_idx,
            "predicted_class": CLASSES[pred_idx],
            "confidence": float(confs[i].item()),
            "top_k": [
                {
                    "class_id": int(top_idxs[i, j].item()),
                    "class_name": CLASSES[int(top_idxs[i, j].item())],
                    "probability": float(top_probs[i, j].item())
                }
                for j in range(top_k)
            ]
        })

    return results[0] if len(results) == 1 else results


# ============================================================
# EJEMPLO DE USO
# ============================================================

# z_fused debe venir de V-JEPA2 T=8 fused mean, shape [1024]
# z_fused = ...

# prediction = predict_action_final(z_fused)
# print(prediction["predicted_class"], prediction["confidence"])

Modelos cargados: 7
